In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Dish-Level Data Leakage Check
This notebook checks whether our train, validation, and test sets have overlapping `dish_id` values.

This matters because if the same dish appears in both training and testing, the model may memorize that dish instead of learning general visual patterns for calorie classification.

In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

## Load Nutrition5k Dataset

For this leakage check, we only need two parts of the Nutrition5k dataset:

1. `dish_nutrition_values.csv` — contains one row per dish with calories and nutrition labels.
2. `imagery/realsense_overhead` — contains the overhead RGB image for each dish.

We do not need `dish_ingredients.csv` or `ingredients_metadata.csv` for this check.

In [2]:
DATASET_DIR = "/kaggle/input/datasets/gillesokhin/nutrition5k-dataset"

nutrition_path = os.path.join(DATASET_DIR, "dish_nutrition_values.csv")
IMAGE_ROOT = os.path.join(DATASET_DIR, "imagery/realsense_overhead")

nutrition_df = pd.read_csv(nutrition_path)

print(nutrition_df.shape)
nutrition_df.head()

(4768, 6)


,dish_id,calories,mass,fat,carb,protein
0,dish_1561662216,300.794281,193.0,12.387489,28.218290,18.633970
1,dish_1562688426,137.569992,88.0,8.256000,5.190000,10.297000
2,dish_1561662054,419.438782,292.0,23.838249,26.351543,25.910593
3,dish_1562008979,382.936646,290.0,22.224644,10.173570,35.345387
4,dish_1560455030,20.590000,103.0,0.148000,4.625000,0.956000


## Match Each Dish with Its RGB Image

The nutrition file gives the labels for each dish, but our model uses images.  
Here, we create an image path for each `dish_id` and check whether the overhead RGB image exists.

In [3]:
def get_rgb_path(dish_id):
    return os.path.join(IMAGE_ROOT, dish_id, "rgb.png")

nutrition_df["image_path"] = nutrition_df["dish_id"].apply(get_rgb_path)
nutrition_df["image_exists"] = nutrition_df["image_path"].apply(os.path.exists)

print("Image availability:")
print(nutrition_df["image_exists"].value_counts())

nutrition_df.head()

Image availability:
image_exists
True     3244
False    1524
Name: count, dtype: int64


,dish_id,calories,mass,fat,carb,protein,image_path,image_exists
0,dish_1561662216,300.794281,193.0,12.387489,28.218290,18.633970,/kaggle/input/datasets/gillesokhin/nutrition5k...,True
1,dish_1562688426,137.569992,88.0,8.256000,5.190000,10.297000,/kaggle/input/datasets/gillesokhin/nutrition5k...,False
2,dish_1561662054,419.438782,292.0,23.838249,26.351543,25.910593,/kaggle/input/datasets/gillesokhin/nutrition5k...,True
3,dish_1562008979,382.936646,290.0,22.224644,10.173570,35.345387,/kaggle/input/datasets/gillesokhin/nutrition5k...,True
4,dish_1560455030,20.590000,103.0,0.148000,4.625000,0.956000,/kaggle/input/datasets/gillesokhin/nutrition5k...,True


## Create Clean Dataset

For this project, we only keep dishes that have:

- a valid image file
- complete nutrition labels: calories, mass, fat, carb, and protein

In [4]:
clean_model_df = nutrition_df.dropna(
    subset=["calories", "mass", "fat", "carb", "protein"]
).copy()

clean_model_df = clean_model_df[clean_model_df["image_exists"] == True].copy()

print("Clean dataset shape:", clean_model_df.shape)
clean_model_df.head()

Clean dataset shape: (3244, 8)


,dish_id,calories,mass,fat,carb,protein,image_path,image_exists
0,dish_1561662216,300.794281,193.0,12.387489,28.218290,18.633970,/kaggle/input/datasets/gillesokhin/nutrition5k...,True
2,dish_1561662054,419.438782,292.0,23.838249,26.351543,25.910593,/kaggle/input/datasets/gillesokhin/nutrition5k...,True
3,dish_1562008979,382.936646,290.0,22.224644,10.173570,35.345387,/kaggle/input/datasets/gillesokhin/nutrition5k...,True
4,dish_1560455030,20.590000,103.0,0.148000,4.625000,0.956000,/kaggle/input/datasets/gillesokhin/nutrition5k...,True
5,dish_1558372433,74.360001,143.0,0.286000,0.429000,20.020000,/kaggle/input/datasets/gillesokhin/nutrition5k...,True


## Create Low, Medium, and High Calorie Classes

Instead of predicting exact calories, our model predicts whether a dish is Low, Medium, or High calorie.

We use quantile-based classes so the three groups are roughly balanced.

In [5]:
clean_model_df["calorie_quantile_class"] = pd.qcut(
    clean_model_df["calories"],
    q=3,
    labels=[0, 1, 2]
).astype(int)

class_names_map = {
    0: "Low",
    1: "Medium",
    2: "High"
}

clean_model_df["calorie_quantile_label"] = clean_model_df["calorie_quantile_class"].map(class_names_map)

print("Class balance:")
print(clean_model_df["calorie_quantile_label"].value_counts())

clean_model_df.head()

Class balance:
calorie_quantile_label
High      1082
Medium    1081
Low       1081
Name: count, dtype: int64


,dish_id,calories,mass,fat,carb,protein,image_path,image_exists,calorie_quantile_class,calorie_quantile_label
0,dish_1561662216,300.794281,193.0,12.387489,28.218290,18.633970,/kaggle/input/datasets/gillesokhin/nutrition5k...,True,1,Medium
2,dish_1561662054,419.438782,292.0,23.838249,26.351543,25.910593,/kaggle/input/datasets/gillesokhin/nutrition5k...,True,2,High
3,dish_1562008979,382.936646,290.0,22.224644,10.173570,35.345387,/kaggle/input/datasets/gillesokhin/nutrition5k...,True,2,High
4,dish_1560455030,20.590000,103.0,0.148000,4.625000,0.956000,/kaggle/input/datasets/gillesokhin/nutrition5k...,True,0,Low
5,dish_1558372433,74.360001,143.0,0.286000,0.429000,20.020000,/kaggle/input/datasets/gillesokhin/nutrition5k...,True,0,Low


## Recreate Train, Validation, and Test Split

Here, we recreate the same type of split used for model training:

- 70% training
- 15% validation
- 15% testing

The split is stratified by calorie class so that Low, Medium, and High dishes are represented fairly in each set.

In [6]:
train_df, temp_df = train_test_split(
    clean_model_df,
    test_size=0.30,
    random_state=42,
    stratify=clean_model_df["calorie_quantile_class"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["calorie_quantile_class"]
)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain class balance:")
print(train_df["calorie_quantile_label"].value_counts())

print("\nValidation class balance:")
print(val_df["calorie_quantile_label"].value_counts())

print("\nTest class balance:")
print(test_df["calorie_quantile_label"].value_counts())

Train: (2270, 10)
Validation: (487, 10)
Test: (487, 10)

Train class balance:
calorie_quantile_label
Medium    757
High      757
Low       756
Name: count, dtype: int64

Validation class balance:
calorie_quantile_label
High      163
Low       162
Medium    162
Name: count, dtype: int64

Test class balance:
calorie_quantile_label
Low       163
Medium    162
High      162
Name: count, dtype: int64


## Check for Dish-Level Data Leakage

To check for data leakage, we compare the `dish_id` values across train, validation, and test sets.

If the same `dish_id` appears in more than one split, then the model may be evaluated on a dish it already saw during training.

In [7]:
train_ids = set(train_df["dish_id"])
val_ids = set(val_df["dish_id"])
test_ids = set(test_df["dish_id"])

train_val_overlap = train_ids & val_ids
train_test_overlap = train_ids & test_ids
val_test_overlap = val_ids & test_ids

print("Train-Val overlap:", len(train_val_overlap))
print("Train-Test overlap:", len(train_test_overlap))
print("Val-Test overlap:", len(val_test_overlap))

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


In [8]:
if len(train_val_overlap) > 0:
    print("Example Train-Val overlapping dish IDs:")
    print(list(train_val_overlap)[:10])

if len(train_test_overlap) > 0:
    print("Example Train-Test overlapping dish IDs:")
    print(list(train_test_overlap)[:10])

if len(val_test_overlap) > 0:
    print("Example Val-Test overlapping dish IDs:")
    print(list(val_test_overlap)[:10])

if (
    len(train_val_overlap) == 0
    and len(train_test_overlap) == 0
    and len(val_test_overlap) == 0
):
    print("No dish-level overlap found across train, validation, and test sets.")

No dish-level overlap found across train, validation, and test sets.


## Result

The overlap count was 0 between train-validation, train-test, and validation-test.

This means the same `dish_id` does not appear across multiple splits. Based on this check, the current overhead RGB image split does not show dish-level data leakage.

In [9]:
leakage_summary = pd.DataFrame({
    "comparison": ["train-val", "train-test", "val-test"],
    "overlap_count": [
        len(train_val_overlap),
        len(train_test_overlap),
        len(val_test_overlap)
    ]
})

leakage_summary.to_csv("/kaggle/working/leakage_check_summary.csv", index=False)
leakage_summary

,comparison,overlap_count
0,train-val,0
1,train-test,0
2,val-test,0


## Final Summary

I checked whether the train, validation, and test sets have overlapping `dish_id` values. The overlap count was 0 for train-validation, train-test, and validation-test.

This suggests that the current split is clean at the dish level for the overhead RGB image setup. Since each dish is represented by one `realsense_overhead/rgb.png` image, the same dish does not appear across multiple splits.

If the team later uses multiple images per dish, such as side-angle images or multiple camera views, we should use a group-based split by `dish_id`.